                # Unoptimized baseline

                The control condition: run the original ChainOfThought program without changing its prompt, demonstrations, or weights.

                **Use it when:** Always measure this first. It tells you whether optimization beats the student you would otherwise deploy.

                **What compilation changes:** Nothing. `compile` is intentionally skipped, so optimization time and cost are zero.

                Important configuration in this benchmark:

                - same task model and frozen test split as every optimized run
- uncached calls so latency and spend remain visible

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'quickstart'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='quickstart'
reading the checked-in chapter result; no API calls


## Compile shape

This is the essential DSPy call used by the shared runner (setup variables omitted):

```python
optimized_detector = detector  # no optimizer and no compile step
```

`compile` returns a program. Calling that program on the untouched test examples is
a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

Unoptimized baseline — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 50.0%
uplift: +0.0 points vs Luna reference
optimization: $0.0000 and 0.0s
inference latency: mean 2.00s; p95 3.50s
reload checks: prompt=True; model=None; predictions=2/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/quickstart.json
- canonical prompt: chapter06/results/final_prompts/quickstart.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

The baseline score is the denominator for uplift. Its contemporaneous predictions are also stored in every optimizer run because uncached model behavior can vary.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.